## AI-Powered Educational Chatbot

In [ ]:
 STEP1 : Importing Library and Data Loading
 STEP2 : Data Cleaning
 STEP3 : Data Embedding and Model Deployment(Transformer)
 STEP4 : Cosine Similarity and Testing Model


## STEP1 : Importing Library and Data Loading 

In [1]:
import numpy as np
import pandas as pd


In [2]:
df = pd.read_csv('knowledge_base1.csv')


In [3]:
df.head()

,id,category,question,answer
0,1,Artificial Intelligence,What is Artificial Intelligence?,Artificial Intelligence (AI) is the field of c...
1,2,Artificial Intelligence,What is an AI agent?,An AI agent is a system that perceives its env...
2,3,Artificial Intelligence,What is supervised AI?,Supervised AI usually refers to AI systems tra...
3,4,Artificial Intelligence,What is weak AI?,"Weak AI, or narrow AI, is designed to perform ..."
4,5,Artificial Intelligence,What is artificial general intelligence?,Artificial General Intelligence (AGI) is a hyp...


In [4]:
df1 = df.copy()

In [5]:
df1.head()

,id,category,question,answer
0,1,Artificial Intelligence,What is Artificial Intelligence?,Artificial Intelligence (AI) is the field of c...
1,2,Artificial Intelligence,What is an AI agent?,An AI agent is a system that perceives its env...
2,3,Artificial Intelligence,What is supervised AI?,Supervised AI usually refers to AI systems tra...
3,4,Artificial Intelligence,What is weak AI?,"Weak AI, or narrow AI, is designed to perform ..."
4,5,Artificial Intelligence,What is artificial general intelligence?,Artificial General Intelligence (AGI) is a hyp...


## STEP2 : Data Cleaning

In [6]:
df1 = df1.drop(['id'],axis = 1)

In [7]:
df1 = df1.drop(['category'],axis = 1)

In [9]:
df1.shape ## It is total shape of whole dataset

(200, 2)

In [10]:
df1.isna().sum() ## It is predicting , there is no null value in this whole dataset.

question    0
answer      0
dtype: int64

In [11]:
df1.info() ## There is information about whole dataset.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  200 non-null    object
 1   answer    200 non-null    object
dtypes: object(2)
memory usage: 3.3+ KB


In [12]:
df1.duplicated().sum()  ## There is no duplicate value

0

In [13]:
import re

# Create a copy of the original dataset
df_clean = df1.copy()

# Convert question and answer to string
df_clean['question'] = df_clean['question'].astype(str)
df_clean['answer'] = df_clean['answer'].astype(str)

# Remove leading and trailing spaces
df_clean['question'] = df_clean['question'].str.strip()
df_clean['answer'] = df_clean['answer'].str.strip()

# Remove HTML tags
df_clean['question'] = df_clean['question'].str.replace(
    r'<[^>]+>', '', regex=True
)

df_clean['answer'] = df_clean['answer'].str.replace(
    r'<[^>]+>', '', regex=True
)

# Replace multiple spaces/newlines with a single space
df_clean['question'] = df_clean['question'].str.replace(
    r'\s+', ' ', regex=True
)

df_clean['answer'] = df_clean['answer'].str.replace(
    r'\s+', ' ', regex=True
)

# Remove duplicate question-answer pairs
df_clean = df_clean.drop_duplicates(
    subset=['question', 'answer']
).reset_index(drop=True)

# Display cleaned data
df_clean.head()

,question,answer
0,What is Artificial Intelligence?,Artificial Intelligence (AI) is the field of c...
1,What is an AI agent?,An AI agent is a system that perceives its env...
2,What is supervised AI?,Supervised AI usually refers to AI systems tra...
3,What is weak AI?,"Weak AI, or narrow AI, is designed to perform ..."
4,What is artificial general intelligence?,Artificial General Intelligence (AGI) is a hyp...


## STEP3 : Data Embedding and Model Deployment(Transformer)

In [14]:
!pip install sentence-transformers



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:

from sentence_transformers import SentenceTransformer

# Load pretrained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all questions
question_embeddings = model.encode(
    df_clean['question'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

# Check embedding shape
print("Embedding Shape:", question_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embedding Shape: (200, 384)


## STEP5 : Testing Model and Cosine Similarity

In [16]:
# Import cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

# Function for semantic question retrieval
def get_answer(user_question, threshold=0.55):

    # Convert user question into embedding
    query_embedding = model.encode(
        user_question,
        normalize_embeddings=True
    )

    # Calculate cosine similarity with all dataset questions
    similarity_scores = cosine_similarity(
        [query_embedding],
        question_embeddings
    )[0]

    # Find the index of the most similar question
    best_index = similarity_scores.argmax()

    # Get the highest similarity score
    best_score = similarity_scores[best_index]

    # Get the matching question and answer
    best_question = df_clean.iloc[best_index]['question']
    best_answer = df_clean.iloc[best_index]['answer']

    # Apply similarity threshold
    if best_score >= threshold:
        return best_question, best_answer, best_score
    else:
        return (
            None,
            "Sorry, I do not have information related to this question.",
            best_score
        )


# -------------------------------
# Test the chatbot
# -------------------------------

user_question = "Tell me something about linear regression."

best_question, answer, score = get_answer(user_question)

print("User Question:", user_question)
print("Best Matching Question:", best_question)
print("Similarity Score:", round(score, 4))
print("Answer:", answer)

User Question: Tell me something about linear regression.
Best Matching Question: What is linear regression?
Similarity Score: 0.8918
Answer: Linear regression models the relationship between input variables and a continuous target using a linear function.


## Hence , I have completed the project.